# Manual llama.cpp CUDA Setup (Cross-Platform)

Run this notebook one time to prepare `llama.cpp` + `llama-server` for GPU usage before running the app.

This notebook does not start the FastAPI app. It prepares tools, validates environment/dependencies, builds with CUDA, and can optionally start `llama-server` for a health check.

Notes:
- Linux: the notebook can attempt `apt-get` installs for missing system deps.
- Windows/macOS: the notebook reports exact missing dependencies with manual install guidance.

In [1]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Python deps used by this notebook (auto-install if missing).
PY_IMPORT_TO_PKG = {
    "requests": "requests",
    "dotenv": "python-dotenv",
    "huggingface_hub": "huggingface-hub",
}

def ensure_python_packages(import_to_pkg: dict[str, str]) -> None:
    missing = []
    for import_name, pip_name in import_to_pkg.items():
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pip_name)

    if not missing:
        print("All notebook Python dependencies are installed.")
        return

    print(f"Installing missing Python packages: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", *missing], check=True)

    # Re-validate imports after installation.
    for import_name in import_to_pkg.keys():
        __import__(import_name)
    print("Python dependency installation verified.")

ensure_python_packages(PY_IMPORT_TO_PKG)

import requests
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download

load_dotenv()

# Notebook is expected to run from App/. Keep workspace-root derived paths stable.
ROOT = Path.cwd().resolve()
WORKSPACE_ROOT = ROOT.parent
LLAMA_CPP_DIR = (WORKSPACE_ROOT / "llama.cpp").resolve()
BUILD_DIR = LLAMA_CPP_DIR / "build"
GGUF_MODELS_DIR = (WORKSPACE_ROOT / "models" / "gguf").resolve()
GGUF_MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Default HF model requested by user
HF_GGUF_REPO_ID = os.getenv("HF_GGUF_REPO_ID", "H4miid/qwen2_5_coder_7b_merged_f16.gguf").strip()
HF_GGUF_FILENAME = os.getenv("HF_GGUF_FILENAME", "").strip()
if not HF_GGUF_FILENAME:
    HF_GGUF_FILENAME = HF_GGUF_REPO_ID.rsplit("/", 1)[-1]

HF_GGUF_REVISION = os.getenv("HF_GGUF_REVISION", "main").strip()
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

print(f"ROOT={ROOT}")
print(f"WORKSPACE_ROOT={WORKSPACE_ROOT}")
print(f"LLAMA_CPP_DIR={LLAMA_CPP_DIR}")
print(f"BUILD_DIR={BUILD_DIR}")
print(f"GGUF_MODELS_DIR={GGUF_MODELS_DIR}")
print(f"HF default model={HF_GGUF_REPO_ID}")

All notebook Python dependencies are installed.
ROOT=/workspace/MentorApp/App
WORKSPACE_ROOT=/workspace/MentorApp
LLAMA_CPP_DIR=/workspace/MentorApp/llama.cpp
BUILD_DIR=/workspace/MentorApp/llama.cpp/build
GGUF_MODELS_DIR=/workspace/MentorApp/models/gguf
HF default model=H4miid/qwen2_5_coder_7b_merged_f16.gguf


/workspace/MentorApp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Strict preflight checks + best-effort system auto-install (Linux apt).
import tempfile
from pathlib import Path

def run_cmd(cmd: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    print("$", " ".join(cmd))
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
    )

def apt_install_if_possible(packages: list[str]) -> None:
    if not packages:
        return
    if sys.platform != "linux":
        print(f"Non-Linux platform. Skipping auto-install for: {packages}")
        return
    apt_get = shutil.which("apt-get")
    if not apt_get:
        print("apt-get not found. Cannot auto-install system packages.")
        return

    is_root = hasattr(os, "geteuid") and os.geteuid() == 0
    sudo = shutil.which("sudo")
    base_cmd = ["apt-get"] if is_root else (["sudo", "-n", "apt-get"] if sudo else None)
    if base_cmd is None:
        print("No root privileges and sudo unavailable. Cannot auto-install system packages.")
        return

    try:
        run_cmd(base_cmd + ["update"], check=True)
        run_cmd(base_cmd + ["install", "-y", *packages], check=True)
        print(f"System package install attempted for: {packages}")
    except subprocess.CalledProcessError:
        print("System package install attempt failed. Continue to strict validation.")

def apt_install_candidates(packages: list[str]) -> None:
    if not packages:
        return
    if sys.platform != "linux":
        return
    apt_get = shutil.which("apt-get")
    if not apt_get:
        return

    is_root = hasattr(os, "geteuid") and os.geteuid() == 0
    sudo = shutil.which("sudo")
    base_cmd = ["apt-get"] if is_root else (["sudo", "-n", "apt-get"] if sudo else None)
    if base_cmd is None:
        return

    run_cmd(base_cmd + ["update"], check=False)
    for pkg in packages:
        print(f"Trying package: {pkg}")
        subprocess.run(base_cmd + ["install", "-y", pkg], check=False, text=True)

def parse_cmake_version() -> tuple[int, int, int] | None:
    try:
        out = subprocess.check_output(["cmake", "--version"], text=True)
        first = out.splitlines()[0].split()[-1]
        parts = first.split(".")
        while len(parts) < 3:
            parts.append("0")
        return int(parts[0]), int(parts[1]), int(parts[2])
    except Exception:
        return None

def detect_cuda_root() -> Path | None:
    candidates = []
    for key in ("CUDAToolkit_ROOT", "CUDA_PATH", "CUDA_HOME"):
        value = os.getenv(key, "").strip()
        if value:
            candidates.append(Path(value).expanduser().resolve())

    # Common Linux locations.
    candidates.extend(
        [
            Path("/usr/local/cuda"),
            Path("/usr/local/cuda-13.1"),
            Path("/usr/local/cuda-13.0"),
            Path("/usr/local/cuda-12.8"),
            Path("/opt/cuda"),
        ]
    )

    seen = set()
    for p in candidates:
        p = p.resolve()
        if str(p) in seen:
            continue
        seen.add(str(p))
        if (p / "bin" / "nvcc").exists():
            return p
    return None

def find_cuda_libs(cuda_root: Path) -> tuple[list[Path], list[Path], list[str]]:
    search_dirs = [
        cuda_root / "lib64",
        cuda_root / "targets" / "x86_64-linux" / "lib",
        Path("/usr/lib/x86_64-linux-gnu"),
        Path("/usr/lib64"),
    ]

    cublas = []
    cublas_lt = []
    for d in search_dirs:
        if not d.exists():
            continue
        cublas.extend(sorted(d.glob("libcublas.so*")))
        cublas_lt.extend(sorted(d.glob("libcublasLt.so*")))

    ldconfig_hits = []
    if sys.platform == "linux" and shutil.which("ldconfig"):
        try:
            out = subprocess.check_output(["ldconfig", "-p"], text=True, stderr=subprocess.STDOUT)
            for line in out.splitlines():
                low = line.lower()
                if "cublas" in low or "cudart" in low:
                    ldconfig_hits.append(line.strip())
        except Exception:
            pass

    return cublas, cublas_lt, ldconfig_hits

def probe_cmake_cuda_targets(cuda_root: Path) -> tuple[bool, str]:
    # Definitive test: can CMake create CUDA::cublas imported target?
    cmake_lists = """
cmake_minimum_required(VERSION 3.18)
project(cuda_probe LANGUAGES C CXX)
find_package(CUDAToolkit REQUIRED)
if(NOT TARGET CUDA::cublas)
  message(FATAL_ERROR "CUDA::cublas target missing")
endif()
message(STATUS "CUDA::cublas target available")
"""
    with tempfile.TemporaryDirectory(prefix="cuda_probe_") as td:
        td_path = Path(td)
        (td_path / "CMakeLists.txt").write_text(cmake_lists, encoding="utf-8")
        proc = subprocess.run(
            [
                "cmake",
                "-S",
                str(td_path),
                "-B",
                str(td_path / "build"),
                f"-DCUDAToolkit_ROOT={cuda_root}",
                f"-DCMAKE_PREFIX_PATH={cuda_root}",
            ],
            text=True,
            capture_output=True,
            check=False,
        )
        output = (proc.stdout or "") + "\n" + (proc.stderr or "")
        return proc.returncode == 0, output

# 1) Ensure base tooling exists (auto-install attempts on Linux).
required_tools = ["git", "cmake", "nvcc"]
missing_tools = [t for t in required_tools if shutil.which(t) is None]
if missing_tools:
    apt_package_map = {
        "git": "git",
        "cmake": "cmake",
        "nvcc": "nvidia-cuda-toolkit",
    }
    apt_install_if_possible([apt_package_map[t] for t in missing_tools if t in apt_package_map])

# Recheck after attempted install.
missing_tools = [t for t in required_tools if shutil.which(t) is None]
if missing_tools:
    raise RuntimeError(
        f"Missing required tools on PATH after install attempt: {missing_tools}. "
        "Install them, then rerun this cell."
    )

print("Required tools found: git, cmake, nvcc")

# 2) Validate CMake version for modern CUDA target resolution.
cmake_version = parse_cmake_version()
print(f"cmake version: {cmake_version}")
if cmake_version is None or cmake_version < (3, 23, 0):
    raise RuntimeError(
        "CMake >= 3.23 is required for reliable CUDA imported targets (CUDA::cublas)."
    )

# 3) Detect CUDA root and validate CMake target resolution.
CUDA_ROOT = detect_cuda_root()
if CUDA_ROOT is None:
    raise RuntimeError(
        "CUDA toolkit root not found. Set CUDAToolkit_ROOT or CUDA_PATH, or install CUDA toolkit."
    )

os.environ["CUDAToolkit_ROOT"] = str(CUDA_ROOT)
os.environ["CUDA_PATH"] = str(CUDA_ROOT)
os.environ["CUDA_HOME"] = str(CUDA_ROOT)
os.environ["PATH"] = f"{CUDA_ROOT / 'bin'}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{CUDA_ROOT / 'lib64'}:{os.environ.get('LD_LIBRARY_PATH', '')}"

print(f"Using CUDA_ROOT={CUDA_ROOT}")
run_cmd(["nvcc", "--version"], check=True)

cublas_libs, cublas_lt_libs, ldconfig_hits = find_cuda_libs(CUDA_ROOT)
print(f"Detected libcublas entries: {len(cublas_libs)}")
print(f"Detected libcublasLt entries: {len(cublas_lt_libs)}")
if ldconfig_hits:
    print("ldconfig CUDA/cuBLAS entries:")
    for line in ldconfig_hits[:8]:
        print("  ", line)

probe_ok, probe_output = probe_cmake_cuda_targets(CUDA_ROOT)
if not probe_ok:
    print("CUDA::cublas probe failed. Attempting package candidates...")
    apt_install_candidates([
        "libcublas-dev-13-1",
        "libcublas-13-1",
        "cuda-cublas-dev-13-1",
        "cuda-cublas-13-1",
        "cuda-toolkit-13-1",
    ])
    probe_ok, probe_output = probe_cmake_cuda_targets(CUDA_ROOT)

if not probe_ok:
    tail = "\n".join(probe_output.splitlines()[-40:])
    raise RuntimeError(
        "CMake cannot resolve CUDA::cublas with the detected toolkit. "
        "Install matching CUDA cuBLAS packages for your toolkit version, then rerun. "
        f"Probe output (tail):\n{tail}"
    )

print("CUDA/cuBLAS preflight passed (CMake CUDA::cublas target is available).")

Required tools found: git, cmake, nvcc
cmake version: (3, 28, 3)
Using CUDA_ROOT=/usr/local/cuda-13.1
$ nvcc --version
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Tue_Dec_16_07:23:41_PM_PST_2025
Cuda compilation tools, release 13.1, V13.1.115
Build cuda_13.1.r13.1/compiler.37061995_0
Detected libcublas entries: 0
Detected libcublasLt entries: 0
ldconfig CUDA/cuBLAS entries:
   libcudart.so.13 (libc6,x86-64) => /usr/local/cuda/lib64/libcudart.so.13
   libcudart.so (libc6,x86-64) => /usr/local/cuda/lib64/libcudart.so
CUDA::cublas probe failed. Attempting package candidates...
$ apt-get update
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package 

debconf: delaying package configuration, since apt-utils is not installed


Fetched 798 MB in 10s (79.3 MB/s)
Selecting previously unselected package libcublas-13-1.
(Reading database ... 42029 files and directories currently installed.)
Preparing to unpack .../libcublas-13-1_13.2.1.1-1_amd64.deb ...
Unpacking libcublas-13-1 (13.2.1.1-1) ...
Selecting previously unselected package libcublas-dev-13-1.
Preparing to unpack .../libcublas-dev-13-1_13.2.1.1-1_amd64.deb ...
Unpacking libcublas-dev-13-1 (13.2.1.1-1) ...
Setting up libcublas-13-1 (13.2.1.1-1) ...
Setting up libcublas-dev-13-1 (13.2.1.1-1) ...
Trying package: libcublas-13-1
Reading package lists...
Building dependency tree...
Reading state information...
libcublas-13-1 is already the newest version (13.2.1.1-1).
libcublas-13-1 set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 19 not upgraded.
Trying package: cuda-cublas-dev-13-1
Reading package lists...
Building dependency tree...
Reading state information...
Trying package: cuda-cublas-13-1
Reading package lists...

E: Unable to locate package cuda-cublas-dev-13-1



Building dependency tree...
Reading state information...
Trying package: cuda-toolkit-13-1
Reading package lists...

E: Unable to locate package cuda-cublas-13-1



Building dependency tree...
Reading state information...
The following additional packages will be installed:
  adwaita-icon-theme at-spi2-common at-spi2-core ca-certificates-java
  cuda-command-line-tools-13-1 cuda-compiler-13-1 cuda-cuobjdump-13-1
  cuda-cupti-13-1 cuda-cupti-dev-13-1 cuda-cuxxfilt-13-1
  cuda-documentation-13-1 cuda-gdb-13-1 cuda-libraries-13-1
  cuda-libraries-dev-13-1 cuda-nsight-13-1 cuda-nsight-compute-13-1
  cuda-nsight-systems-13-1 cuda-nvdisasm-13-1 cuda-nvml-dev-13-1
  cuda-nvprune-13-1 cuda-nvrtc-dev-13-1 cuda-nvtx-13-1 cuda-opencl-13-1
  cuda-profiler-api-13-1 cuda-sandbox-dev-13-1 cuda-sanitizer-13-1
  cuda-tileiras-13-1 cuda-tools-13-1 cuda-visual-tools-13-1 dbus-user-session
  dconf-gsettings-backend dconf-service default-jre default-jre-headless
  gds-tools-13-1 gsettings-desktop-schemas gtk-update-icon-cache
  hicolor-icon-theme humanity-icon-theme java-common libatk-bridge2.0-0t64
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0t64 libatspi

debconf: delaying package configuration, since apt-utils is not installed


Fetched 2396 MB in 2min 10s (18.5 MB/s)
Selecting previously unselected package dbus-user-session.
(Reading database ... 42061 files and directories currently installed.)
Preparing to unpack .../000-dbus-user-session_1.14.10-4ubuntu4.1_amd64.deb ...
Unpacking dbus-user-session (1.14.10-4ubuntu4.1) ...
Selecting previously unselected package libxmuu1:amd64.
Preparing to unpack .../001-libxmuu1_2%3a1.1.3-3build2_amd64.deb ...
Unpacking libxmuu1:amd64 (2:1.1.3-3build2) ...
Selecting previously unselected package gtk-update-icon-cache.
Preparing to unpack .../002-gtk-update-icon-cache_3.24.41-4ubuntu1.3_amd64.deb ...
Unpacking gtk-update-icon-cache (3.24.41-4ubuntu1.3) ...
Selecting previously unselected package hicolor-icon-theme.
Preparing to unpack .../003-hicolor-icon-theme_0.17-2_all.deb ...
Unpacking hicolor-icon-theme (0.17-2) ...
Selecting previously unselected package humanity-icon-theme.
Preparing to unpack .../004-humanity-icon-theme_0.6.16_all.deb ...
Unpacking humanity-icon-th

In [6]:
# Clone llama.cpp and build it with CUDA support (idempotent single flow).
# This replaces duplicated clone/build cells below.


# Step 1: Clone the repo (skip if it already exists)
if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp ...")
    run_cmd(["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)], check=True)
    print(f"Cloned -> {LLAMA_CPP_DIR}")
else:
    print(f"llama.cpp already at {LLAMA_CPP_DIR}")

# Step 2: Configure + build with explicit CUDA hints
BUILD_DIR.mkdir(parents=True, exist_ok=True)
default_generator = "Ninja" if shutil.which("ninja") else "Unix Makefiles"
generator = os.getenv("CMAKE_GENERATOR", default_generator)

# Reuse existing generator from CMake cache to avoid generator mismatch on reruns.
cache_file = BUILD_DIR / "CMakeCache.txt"
if cache_file.exists():
    try:
        cache_text = cache_file.read_text(encoding="utf-8", errors="ignore")
        cache_generator = None
        for line in cache_text.splitlines():
            if line.startswith("CMAKE_GENERATOR:INTERNAL="):
                cache_generator = line.split("=", 1)[1].strip()
                break
        if cache_generator and cache_generator != generator:
            print(f"Reusing cached generator: {cache_generator} (requested: {generator})")
            generator = cache_generator
    except Exception:
        pass

cmake_configure_cmd = [
    "cmake",
    "-S",
    str(LLAMA_CPP_DIR),
    "-B",
    str(BUILD_DIR),
    "-G",
    generator,
    "-DGGML_CUDA=ON",
    "-DCMAKE_BUILD_TYPE=Release",
    f"-DCUDAToolkit_ROOT={CUDA_ROOT}",
    f"-DCMAKE_PREFIX_PATH={CUDA_ROOT}",
]

nvcc_path = CUDA_ROOT / "bin" / "nvcc"
if nvcc_path.exists():
    cmake_configure_cmd.append(f"-DCMAKE_CUDA_COMPILER={nvcc_path}")

print("Configuring CMake with CUDA ...")
run_cmd(cmake_configure_cmd, check=True)

build_jobs = str(max(1, (os.cpu_count() or 4) - 1))
cmake_build_cmd = [
    "cmake",
    "--build",
    str(BUILD_DIR),
    "--config",
    "Release",
    "-j",
    build_jobs,
]

print("Building llama.cpp (this can take several minutes) ...")
run_cmd(cmake_build_cmd, check=True)

# Step 3: Locate llama-server executable deterministically
candidates = [
    BUILD_DIR / "bin" / "Release" / "llama-server.exe",  # Windows MSVC
    BUILD_DIR / "bin" / "llama-server.exe",                # Windows Ninja
    BUILD_DIR / "bin" / "llama-server",                    # Linux/macOS
    BUILD_DIR / "Release" / "llama-server.exe",
    BUILD_DIR / "llama-server",
]
llama_server_exe = next((p for p in candidates if p.exists() and p.is_file()), None)

if llama_server_exe is None:
    hits = sorted(
        [p for p in BUILD_DIR.rglob("llama-server*") if p.is_file()],
        key=lambda p: (len(p.parts), p.name),
    )
    llama_server_exe = hits[0] if hits else None

if llama_server_exe is None:
    raise FileNotFoundError(
        "llama-server not found after build. Check CMake output, CUDA toolkit, and cuBLAS visibility."
    )

LLAMA_SERVER_EXE = llama_server_exe
print(f"llama-server executable: {llama_server_exe}")

llama.cpp already at /workspace/MentorApp/llama.cpp
Reusing cached generator: Unix Makefiles (requested: Ninja)
Configuring CMake with CUDA ...
$ cmake -S /workspace/MentorApp/llama.cpp -B /workspace/MentorApp/llama.cpp/build -G Unix Makefiles -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DCUDAToolkit_ROOT=/usr/local/cuda-13.1 -DCMAKE_PREFIX_PATH=/usr/local/cuda-13.1 -DCMAKE_CUDA_COMPILER=/usr/local/cuda-13.1/bin/nvcc


CMAKE_BUILD_TYPE=Release


-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- CUDA Toolkit found
-- Replacing 120-real in CMAKE_CUDA_ARCHITECTURES_NATIVE with 120a-real
-- Using CMAKE_CUDA_ARCHITECTURES=120a-real CMAKE_CUDA_ARCHITECTURES_NATIVE=120a-real
-- CUDA host compiler is GNU 13.3.0
-- Including CUDA backend
-- ggml version: 0.9.11
-- ggml commit:  0988accf8
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: common
-- Configuring done (0.5s)
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info


You have changed variables that require your cache to be deleted.
Configure will be re-run and you may have to reset some variables.
The following variables have changed:
CMAKE_CUDA_COMPILER= /usr/local/cuda-13.1/bin/nvcc



-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.43.0") 


CMAKE_BUILD_TYPE=Release


-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5") 
-- Found OpenMP_CXX: -fopenmp (found version "4.5") 
-- Found OpenMP: TRUE (found version "4.5")  
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.9.11
-- ggml commit:  0988accf8
-- Found OpenSSL: /usr/lib/x86_64-linux-gnu/libcrypto.so (found version "3.0.13")  
-- Performing Test OPENSSL_VERSION_SUPPORTED
-- Performing Test OPENSSL_VERSION_SUPPORTED - Success
-- OpenSSL found: 3.0.13
-- Generating embedded license file for target: common
-- Configuring done (1.6s)
-- Generating done (0.3s)
-- Build f

## Optional: Resolve GGUF model and verify llama-server

This section supports both options:
- `LOCAL_GGUF_PATH` in `.env` (direct local file path)
- HF download fallback (default: `H4miid/qwen2_5_coder_7b_merged_f16.gguf`)

Optional HF auth in `.env`:
- `HF_TOKEN` or `HUGGINGFACEHUB_API_TOKEN`

In [9]:
# Resolve GGUF model path (local first, then HF download fallback)
def resolve_gguf_model_path() -> Path:
    local_path = os.getenv("LOCAL_GGUF_PATH", "").strip()
    if local_path:
        raw = Path(local_path).expanduser()
        # Interpret relative LOCAL_GGUF_PATH from workspace root for consistency.
        p = raw if raw.is_absolute() else (WORKSPACE_ROOT / raw).resolve()
        if p.exists():
            print(f"Using LOCAL_GGUF_PATH: {p}")
            return p
        print(f"LOCAL_GGUF_PATH not found, falling back to HF download: {p}")

    target = GGUF_MODELS_DIR / HF_GGUF_FILENAME
    if target.exists():
        print(f"Using cached HF model: {target}")
        return target

    print(f"Downloading HF model: {HF_GGUF_REPO_ID} / {HF_GGUF_FILENAME}")
    downloaded = hf_hub_download(
        repo_id=HF_GGUF_REPO_ID,
        filename=HF_GGUF_FILENAME,
        revision=HF_GGUF_REVISION,
        token=HF_TOKEN,
        local_dir=str(GGUF_MODELS_DIR),
        local_dir_use_symlinks=False,
    )
    p = Path(downloaded).resolve()
    print(f"Downloaded GGUF: {p}")
    return p

GGUF_MODEL_PATH = resolve_gguf_model_path()
print(f"Resolved GGUF_MODEL_PATH={GGUF_MODEL_PATH}")

LOCAL_GGUF_PATH not found, falling back to HF download: /workspace/MentorApp/App/models/gguf/qwen2_5_coder_7b_merged_f16.gguf


/workspace/MentorApp/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Downloaded GGUF: /workspace/MentorApp/models/gguf/qwen2_5_coder_7b_merged_f16.gguf
Resolved GGUF_MODEL_PATH=/workspace/MentorApp/models/gguf/qwen2_5_coder_7b_merged_f16.gguf


In [ ]:
# Optional runtime check
HOST = os.getenv("LLAMA_SERVER_HOST", "127.0.0.1").strip()
PORT = int(os.getenv("LLAMA_SERVER_PORT", "8081"))

if not str(GGUF_MODEL_PATH).strip():
    print("GGUF_MODEL_PATH is empty. Resolve model path first.")
else:
    cmd = [
        str(llama_server_exe),
        "--model",
        str(GGUF_MODEL_PATH),
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--ctx-size",
        "8192",
        "--n-gpu-layers",
        "999",
        "--threads",
        str(max(1, (os.cpu_count() or 4) - 1)),
    ]
    print("Starting llama-server for health check ...")
    print(" ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    health_ok = False
    for _ in range(90):
        time.sleep(1)
        try:
            r = requests.get(f"http://{HOST}:{PORT}/health", timeout=2)
            if r.status_code == 200:
                health_ok = True
                break
        except Exception:
            pass

    print("Health check:", "OK" if health_ok else "FAILED")
    if health_ok:
        print("Set these in .env for app runtime:")
        print(f"LLAMA_SERVER_URL=http://{HOST}:{PORT}/v1")
        print("LLAMA_OPENAI_MODEL=local-gguf")
        print(f"LOCAL_GGUF_PATH={GGUF_MODEL_PATH}")

    # Stop test process. Comment out if you want to keep it running.
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except Exception:
        proc.kill()